# 08 - Fundamentos de MLP (Multi-Layer Perceptron)

## Fase 4: Aprofundamento em Deep Learning

Este notebook explora os **fundamentos teóricos e práticos de MLP**:
- Arquitetura neural (neurônios, camadas, pesos)
- Funções de ativação
- Forward pass (propagação pra frente)
- Implementação em NumPy (educacional)
- Comparação com PyTorch

**Objetivo:** Entender profundamente como MLP funciona, não apenas usá-lo.

## 1. Imports e Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

np.random.seed(42)

print("✓ Setup concluído")

## 2. O que é um Neurônio?

Um **neurônio artificial** é uma unidade de computação que:
1. Recebe múltiplos inputs (x₁, x₂, ..., xₙ)
2. Multiplica cada input por um peso (w₁, w₂, ..., wₙ)
3. Soma tudo + bias (b)
4. Aplica função de ativação
5. Produz output (ŷ)

**Fórmula:** z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
**Output:** ŷ = σ(z) onde σ é função de ativação

In [ ]:
# Visualizar um neurônio
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

# Inputs
inputs = [(1, 5), (1, 3), (1, 1)]
for i, (x, y) in enumerate(inputs):
    circle = plt.Circle((x, y), 0.2, color='lightblue', ec='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(x-0.5, y, f'x₁', fontsize=11, fontweight='bold')

# Pesos
for i, (x, y) in enumerate(inputs):
    mid_x = (x + 4) / 2
    mid_y = (y + 3) / 2
    arrow = FancyArrowPatch((x+0.2, y), (4-0.3, 3), 
                            arrowstyle='->', mutation_scale=20, linewidth=2, color='gray')
    ax.add_patch(arrow)
    ax.text(mid_x, mid_y+0.3, f'w{i+1}', fontsize=10, style='italic')

# Neurônio
neuron = plt.Circle((4, 3), 0.4, color='gold', ec='black', linewidth=2)
ax.add_patch(neuron)
ax.text(4, 3, 'Σ', fontsize=14, fontweight='bold', ha='center', va='center')

# Bias
bias_circle = plt.Circle((4, 1), 0.2, color='lightgreen', ec='black', linewidth=2)
ax.add_patch(bias_circle)
ax.text(4-0.5, 1, 'b', fontsize=11, fontweight='bold')
bias_arrow = FancyArrowPatch((4, 1.2), (4, 2.6), 
                             arrowstyle='->', mutation_scale=20, linewidth=2, color='green')
ax.add_patch(bias_arrow)

# Função de ativação
activation_box = FancyBboxPatch((4-0.5, 2.2), 1, 0.6,
                                boxstyle="round,pad=0.1",
                                edgecolor='black', facecolor='lightyellow', linewidth=2)
ax.add_patch(activation_box)
ax.text(4, 2.5, 'σ(z)', fontsize=10, ha='center', va='center', fontweight='bold')

# Output
output_circle = plt.Circle((7, 3), 0.3, color='lightcoral', ec='black', linewidth=2)
ax.add_patch(output_circle)
ax.text(7-0.5, 3, 'ŷ', fontsize=11, fontweight='bold')
output_arrow = FancyArrowPatch((4.4, 3), (6.7, 3), 
                               arrowstyle='->', mutation_scale=20, linewidth=2, color='red')
ax.add_patch(output_arrow)

ax.set_title('Neurônio Artificial: Computação Básica', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('results/plots/neuron_basic.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📐 EQUAÇÃO DO NEURÔNIO:")
print("\n  z = w₁x₁ + w₂x₂ + w₃x₃ + b")
print("  ŷ = σ(z)")
print("\nOnde:")
print("  x₁, x₂, x₃ = Inputs")
print("  w₁, w₂, w₃ = Pesos (aprendidos)")
print("  b = Bias (aprendido)")
print("  σ = Função de ativação (não-linear)")
print("  ŷ = Output (predição)")

## 3. Funções de Ativação

In [ ]:
# Implementar funções de ativação
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    return np.maximum(0, z)

def tanh(z):
    return np.tanh(z)

def linear(z):
    return z

# Plotar
z = np.linspace(-5, 5, 100)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Sigmoid
ax = axes[0, 0]
ax.plot(z, sigmoid(z), linewidth=2.5, color='blue')
ax.grid(True, alpha=0.3)
ax.set_title('Sigmoid: σ(z) = 1/(1+e^-z)', fontweight='bold')
ax.set_ylabel('Output')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.text(-4, 0.9, 'Range: (0, 1)\nUsado em: Classificação binária', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat'))

# ReLU
ax = axes[0, 1]
ax.plot(z, relu(z), linewidth=2.5, color='green')
ax.grid(True, alpha=0.3)
ax.set_title('ReLU: max(0, z)', fontweight='bold')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.text(-4, 4, 'Range: [0, ∞)\nUsado em: Camadas ocultas\nMais rápido que Sigmoid', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightgreen'))

# Tanh
ax = axes[1, 0]
ax.plot(z, tanh(z), linewidth=2.5, color='red')
ax.grid(True, alpha=0.3)
ax.set_title('Tanh: tanh(z)', fontweight='bold')
ax.set_ylabel('Output')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.text(-4, 0.8, 'Range: (-1, 1)\nUsado em: RNNs, LSTMs', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Linear (sem ativação)
ax = axes[1, 1]
ax.plot(z, linear(z), linewidth=2.5, color='purple')
ax.grid(True, alpha=0.3)
ax.set_title('Linear (sem ativação): z', fontweight='bold')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.text(-4, 4, 'Range: (-∞, ∞)\nUsado em: Camada output\n(regressão)', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue'))

plt.suptitle('Funções de Ativação Comuns', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('results/plots/activation_functions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️  POR QUE NÃO LINEAR?")
print("\nSem função de ativação não-linear:")
print("  • Múltiplas camadas lineares = 1 camada linear")
print("  • Modelo não consegue aprender funções complexas")
print("  • Exemplo: f(f(x)) = f(x) se f é linear")
print("\nCom função de ativação:")
print("  • Cada camada adiciona não-linearidade")
print("  • Modelo consegue aprender qualquer função (aproximação universal)")

## 4. Arquitetura MLP

In [ ]:
# Exemplo de MLP para classificação de imagens (64x64 = 4096 pixels)

class MLPSimple:
    """Implementação educacional de MLP em NumPy"""
    
    def __init__(self, input_size=4096, hidden_sizes=[512, 256], output_size=10):
        """
        Args:
            input_size: Número de features de entrada (pixels)
            hidden_sizes: Lista com tamanho de cada camada oculta
            output_size: Número de classes (output)
        """
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.output_size = output_size
        
        # Inicializar pesos e biases
        self.layers = []
        
        # Camada 1: input -> primeira camada oculta
        layer1 = {
            'W': np.random.randn(input_size, hidden_sizes[0]) * 0.01,
            'b': np.zeros((1, hidden_sizes[0]))
        }
        self.layers.append(layer1)
        
        # Camada 2: primeira oculta -> segunda oculta
        layer2 = {
            'W': np.random.randn(hidden_sizes[0], hidden_sizes[1]) * 0.01,
            'b': np.zeros((1, hidden_sizes[1]))
        }
        self.layers.append(layer2)
        
        # Camada 3: segunda oculta -> output
        layer3 = {
            'W': np.random.randn(hidden_sizes[1], output_size) * 0.01,
            'b': np.zeros((1, output_size))
        }
        self.layers.append(layer3)
    
    def contar_parametros(self):
        """Contar total de parâmetros treináveis"""
        total = 0
        for i, layer in enumerate(self.layers):
            w_params = layer['W'].size
            b_params = layer['b'].size
            layer_params = w_params + b_params
            total += layer_params
            print(f"  Camada {i+1}: {layer['W'].shape} pesos + {layer['b'].shape} bias = {layer_params:,} parâmetros")
        return total

# Criar MLP para MNIST
print("\n📊 ARQUITETURA MLP PARA MNIST (classificação 10 dígitos)")
print("="*60)
print("\nInput: Imagem 64x64 (4.096 pixels)")
print("Camadas ocultas: [512, 256] neurônios")
print("Output: 10 classes (dígitos 0-9)")
print("\nEstrutura:")
print("  4096 (input)")
print("    ↓ [W: 4096×512, b: 512] + ReLU")
print("  512 (hidden 1)")
print("    ↓ [W: 512×256, b: 256] + ReLU")
print("  256 (hidden 2)")
print("    ↓ [W: 256×10, b: 10] + Softmax")
print("  10 (output)")

mlp = MLPSimple(input_size=4096, hidden_sizes=[512, 256], output_size=10)

print("\nParâmetros treináveis:")
total_params = mlp.contar_parametros()
print(f"\n  TOTAL: {total_params:,} parâmetros")
print(f"\nPara comparação:")
print(f"  • CNN MobileNetV2: ~3.5 milhões")
print(f"  • MLP (este modelo): ~{total_params/1e6:.1f} milhões")
print(f"  • Razão: MLP tem {total_params / 3.5e6:.1f}x mais parâmetros que CNN!")

## 5. Forward Pass (Propagação pra Frente)

In [ ]:
def forward_pass(X, mlp):
    """
    Computar forward pass através da rede
    
    X: batch de entrada [batch_size, input_size]
    mlp: modelo MLP com pesos e bias
    
    Retorna: predições [batch_size, output_size]
    """
    
    # Camada 1: input -> hidden 1 + ReLU
    z1 = X @ mlp.layers[0]['W'] + mlp.layers[0]['b']
    a1 = np.maximum(0, z1)  # ReLU
    
    # Camada 2: hidden 1 -> hidden 2 + ReLU
    z2 = a1 @ mlp.layers[1]['W'] + mlp.layers[1]['b']
    a2 = np.maximum(0, z2)  # ReLU
    
    # Camada 3: hidden 2 -> output + Softmax
    z3 = a2 @ mlp.layers[2]['W'] + mlp.layers[2]['b']
    # Softmax
    exp_z3 = np.exp(z3 - np.max(z3, axis=1, keepdims=True))  # Numerical stability
    a3 = exp_z3 / np.sum(exp_z3, axis=1, keepdims=True)
    
    return a3, (z1, a1, z2, a2, z3)  # retornar também intermediários para backprop

# Testar forward pass
print("\n🔄 FORWARD PASS (Propagação pra frente)")
print("="*60)

# Criar batch de teste (4 imagens)
batch_size = 4
X_test = np.random.randn(batch_size, 4096)  # 4 imagens de 64x64

print(f"\nInput shape: {X_test.shape}")
print(f"Meaning: {batch_size} imagens, cada uma com 4.096 pixels")

# Forward pass
predictions, cache = forward_pass(X_test, mlp)

print(f"\nOutput shape: {predictions.shape}")
print(f"Meaning: {batch_size} predições, 10 probabilidades cada (softmax)")

print(f"\nExemplo - Predição para primeira imagem:")
print(f"Classe 0: {predictions[0, 0]:.4f}")
print(f"Classe 1: {predictions[0, 1]:.4f}")
print(f"Classe 2: {predictions[0, 2]:.4f}")
print(f"...")
print(f"Classe 9: {predictions[0, 9]:.4f}")
print(f"\nSoma (deve ser ~1.0): {predictions[0].sum():.6f}")
print(f"Classe predita: {np.argmax(predictions[0])} com confiança {np.max(predictions[0]):.2%}")

## 6. Visualizar Transformação de Dados Através das Camadas

In [ ]:
# Visualizar como dados mudam através das camadas

print("\n📈 TRANSFORMAÇÃO DE DADOS ATRAVÉS DAS CAMADAS")
print("="*60)

print(f"\nCamada 0 (Input):")
print(f"  Shape: (batch=4, features=4096)")
print(f"  Significado: 4 imagens, cada uma com 4.096 pixels")
print(f"  Valores: pixels da imagem (0-255 normalizado)")

z1, a1, z2, a2, z3 = cache

print(f"\nCamada 1 (Hidden - antes ReLU):")
print(f"  Shape: {z1.shape}")
print(f"  Significado: 4 amostras, 512 features abstratas")
print(f"  Valores: z = X @ W + b (pode ser negativo)")
print(f"  Min: {z1.min():.2f}, Max: {z1.max():.2f}")

print(f"\nCamada 1 (Hidden - depois ReLU):")
print(f"  Shape: {a1.shape}")
print(f"  Significado: 4 amostras, 512 features (apenas positivos)")
print(f"  Valores: max(0, z) (supressão de negativos)")
print(f"  Min: {a1.min():.2f}, Max: {a1.max():.2f}")

print(f"\nCamada 2 (Hidden - antes ReLU):")
print(f"  Shape: {z2.shape}")
print(f"  Significado: 4 amostras, 256 features")

print(f"\nCamada 2 (Hidden - depois ReLU):")
print(f"  Shape: {a2.shape}")

print(f"\nCamada 3 (Output - logits):")
print(f"  Shape: {z3.shape}")
print(f"  Significado: 4 amostras, 10 classes")

print(f"\nCamada 3 (Output - softmax):")
print(f"  Shape: {predictions.shape}")
print(f"  Significado: Probabilidades (soma = 1.0)")

# Visualizar
fig, axes = plt.subplots(2, 3, figsize=(14, 6))

axes[0, 0].hist(X_test.flatten(), bins=50, color='blue', alpha=0.7)
axes[0, 0].set_title('Input (pixels)', fontweight='bold')
axes[0, 0].set_xlabel('Value')

axes[0, 1].hist(a1.flatten(), bins=50, color='green', alpha=0.7)
axes[0, 1].set_title('Hidden 1 (512 features)', fontweight='bold')
axes[0, 1].set_xlabel('Value')

axes[0, 2].hist(a2.flatten(), bins=50, color='orange', alpha=0.7)
axes[0, 2].set_title('Hidden 2 (256 features)', fontweight='bold')
axes[0, 2].set_xlabel('Value')

axes[1, 0].imshow(X_test[0].reshape(64, 64), cmap='gray')
axes[1, 0].set_title('Imagem de entrada', fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].bar(range(512), a1[0, :], color='green', alpha=0.7)
axes[1, 1].set_title('Ativações Hidden 1 (1ª imagem)', fontweight='bold')
axes[1, 1].set_xlabel('Neurônio')
axes[1, 1].set_ylabel('Ativação')

axes[1, 2].bar(range(10), predictions[0], color='red', alpha=0.7)
axes[1, 2].set_title('Probabilidades Output', fontweight='bold')
axes[1, 2].set_xlabel('Classe')
axes[1, 2].set_ylabel('Probabilidade')
axes[1, 2].set_xticks(range(10))

plt.suptitle('Transformação de Dados Através da MLP', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/mlp_data_transformation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Resumo e Conclusão

In [ ]:
print("\n" + "="*70)
print("RESUMO: FUNDAMENTOS DE MLP")
print("="*70)

print("""
1. NEURÔNIO ARTIFICIAL:
   • Combina inputs com pesos (w) e bias (b)
   • z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
   • Aplica função de ativação: ŷ = σ(z)
   • Parâmetro a aprender: W e b

2. FUNÇÕES DE ATIVAÇÃO:
   • Sigmoid: Saída (0,1), usado em classificação binária
   • ReLU: Saída [0,∞), mais rápido, usado em camadas ocultas
   • Tanh: Saída (-1,1), usado em RNNs
   • Linear: Sem ativação, usado em outputs de regressão
   • SoftMax: Probabilidades (soma=1), usado em multi-classe

3. ARQUITETURA MLP:
   • Camadas conectadas: cada neurônio conectado a TODOS da camada anterior
   • Totalmente conectada (fully connected / dense)
   • Input → Hidden 1 → Hidden 2 → ... → Output
   • Exemplo: 4096 → 512 → 256 → 10

4. FORWARD PASS:
   • Passa dados através de cada camada sequencialmente
   • Cada camada: z = X @ W + b, depois aplica ativação
   • Resultado: predição (probabilidades)
   • Tempo: ~1ms para 1 imagem em CPU

5. PARÂMETROS:
   • MLP para MNIST: ~2.1 milhões de parâmetros
   • CNN MobileNetV2: ~3.5 milhões
   • MLP tem MAIS parâmetros mas CNN é mais eficiente!

6. POR QUE CNN É MELHOR PARA IMAGENS:
   ✗ MLP: Trata cada pixel independentemente (perde estrutura espacial)
   ✗ MLP: Precisa de MUITO mais parâmetros
   ✗ MLP: Lento para imagens grandes
   
   ✓ CNN: Usa convolução (detecta padrões locais)
   ✓ CNN: Compartilha pesos (menos parâmetros)
   ✓ CNN: Eficiente para imagens
""")

print("="*70)
print("PRÓXIMO NOTEBOOK: Backpropagation - Como os pesos são atualizados?")
print("="*70)